# Music Annotation Project
## Notebook 89 — A5 Multi-Threshold High-Resolution Visual Audit

**CPU runtime.** This notebook reads a completed Notebook 88 run. It does not retrain, refit or recalibrate A5.

Outputs use **green = annotation correctly cleaned**, **red = genuine print lost**, and **purple = truth uncertain**. Exact masks and enlarged review overlays are saved separately.


In [2]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow matplotlib


Mounted at /content/drive


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, math, shutil, sys, platform, zipfile
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
SOURCE_RUN_OVERRIDE=None
N88_ROOT=ROOT/'a5_contextual_safe_cleanup_v88'
OUTPUT_ROOT=ROOT/'notebook_89_a5_visual_audit'
THRESHOLDS=[0.995,0.990,0.980,0.950]
MIN_CLEAN_RUN=2
REVIEW_RADIUS=3
OVERLAY_ALPHA=.72
CROP_PADDING=120
MIN_CROP_SIDE=520
MAX_CROP_SIDE=1400
MAX_QUEUE=50

RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_notebook89_a5_visual_audit'
OUT=OUTPUT_ROOT/RUN; TABLES=OUT/'tables'; CROPS=OUT/'crops'; WHOLE=OUT/'whole_pages'; VISUAL=OUT/'visual_audit'; AUDIT=OUT/'audit'; UPLOAD=OUT/'upload'
for p in [OUT,TABLES,CROPS,WHOLE,VISUAL,AUDIT,UPLOAD]: p.mkdir(parents=True,exist_ok=False)

def require(x,msg):
    if not x: raise RuntimeError(msg)

def sha256_file(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()

def load_gray(path):
    path = Path(path)
    if path.suffix.lower() == '.npy':
        image = np.load(path,allow_pickle=False,)
        require(image.ndim == 2,
            (
                f"{path}: expected 2D cached array, "
                f"got shape {image.shape}"
            ),
        )
        if image.dtype == np.bool_:
            image = (image.astype(np.uint8) * 255)
        elif image.dtype != np.uint8:
            require(
                np.isfinite(image).all(),
                (
                    f"{path}: non-finite image values"
                ),
            )
            require(
                image.min() >= 0
                and image.max() <= 255,
                (
                    f"{path}: grayscale values outside "
                    "[0,255]"
                ),
            )
            image = image.astype(np.uint8)
        return image

    image = cv2.imread(str(path),cv2.IMREAD_GRAYSCALE)

    require(image is not None,
    (
            f"Could not read image: {path}"
        ),
    )
    return image

print('Output:',OUT)


Output: /content/drive/MyDrive/MusicAnnotationProject/notebook_89_a5_visual_audit/20261123_095620_notebook89_a5_visual_audit


## 1. Resolve and validate the completed Notebook 88 run


In [ ]:
PRED_NAMES=['a5_primary_oof_predictions.parquet','all_eligible_oof_predictions.parquet','a5_all_eligible_oof_predictions.parquet','oof_predictions.parquet']

def candidates(root):
    hits=[]
    for name in PRED_NAMES:
        for f in root.rglob(name):
            hits.extend([f.parent,f.parent.parent])
    out=[]; seen=set()
    for p in hits:
        if str(p) in seen: continue
        seen.add(str(p))
        if any((p/'tables'/n).exists() or (p/n).exists() for n in PRED_NAMES): out.append(p)
    return sorted(out,key=lambda p:p.stat().st_mtime)

if SOURCE_RUN_OVERRIDE:
    SOURCE=Path(SOURCE_RUN_OVERRIDE)
else:
    found=candidates(N88_ROOT)
    require(found,'No completed Notebook 88 run found. Set SOURCE_RUN_OVERRIDE to the exact run directory.')
    SOURCE=found[-1]

def find_file(names):
    for n in names:
        for p in [SOURCE/'tables'/n,SOURCE/n]:
            if p.exists(): return p
    return None
PRED=find_file(PRED_NAMES)
require(PRED is not None,'Notebook 88 prediction parquet not found.')
full=pd.read_parquet(PRED)

aliases={
    'page':['page','page_id','stem'], 'source_group':['source_group','group'],
    'y':['y','row','pixel_y'], 'x':['x','col','pixel_x'],
    'truth':['truth','label','truth_class'],
    'prob':['p_annotation_a5','p_clean','clean_probability','prob_clean','calibrated_clean_probability','p_annotation','annotation_probability']}
resolved={k:next((x for x in v if x in full.columns),None) for k,v in aliases.items()}
require(all(resolved[k] for k in ['page','y','x','truth','prob']),f'Could not resolve prediction schema. Columns: {list(full.columns)}')
full=full.rename(columns={v:k for k,v in resolved.items() if v and v!=k}).copy()
if 'source_group' not in full: full['source_group']='UNKNOWN'
for x in ['y','x','truth']: full[x]=pd.to_numeric(full[x],errors='raise').astype(int)
full['prob']=pd.to_numeric(full.prob,errors='raise').astype(float)
full['page']=full.page.astype(str); full['source_group']=full.source_group.astype(str)
require(np.isfinite(full.prob).all(),'Non-finite probabilities found.')
require(not full.duplicated(['page','y','x']).any(),'Duplicate page/y/x rows found.')
require(set(full.truth.unique()).issubset({0,1,2}),'Expected truth labels 0=ANNOTATION, 1=PRINT, 2=UNCERTAIN.')
(AUDIT/'schema_resolution.json').write_text(json.dumps({'source':str(SOURCE),'predictions':str(PRED),'resolved':resolved,'rows':len(full)},indent=2))
print('Source:',SOURCE,'Rows:',len(full),'Pages:',full.page.nunique())


Source: /content/drive/MyDrive/MusicAnnotationProject/a5_contextual_safe_cleanup_v88/20261122_155756_notebook88_a5_training Rows: 324592 Pages: 56


## 2. Locate observed, clean-truth and A1 images


In [ ]:
SUP = {
    '.png',
    '.jpg',
    '.jpeg',
    '.tif',
    '.tiff',
    '.bmp',
}


def imap(folder):

    folder = Path(
        folder
    )

    return {
        p.stem: p

        for p
        in folder.iterdir()

        if (
            p.is_file()
            and p.suffix.lower()
            in SUP
        )
    } if folder.exists() else {}


obsmap = imap(
    ROOT
    / 'aligned'
)

cleanmap = imap(
    ROOT
    / 'source'
    / 'clean'
)


# ------------------------------------------------------------
# Resolve Notebook 88 observable-channel cache
# ------------------------------------------------------------

CHANNEL_CACHE_CANDIDATES = [
    SOURCE
    / 'observable_channel_cache',

    SOURCE
    / 'channel_cache',

    SOURCE
    / 'cache'
    / 'observable_channels',

    SOURCE
    / 'cache',
]


CHANNEL_CACHE = next(
    (
        p

        for p
        in CHANNEL_CACHE_CANDIDATES

        if p.exists()
        and p.is_dir()
    ),
    None,
)


require(
    CHANNEL_CACHE is not None,
    (
        "Could not locate Notebook 88 channel cache. "
        "Tried:\n"
        + "\n".join(
            str(p)

            for p
            in CHANNEL_CACHE_CANDIDATES
        )
    ),
)


print(
    'Notebook 88 channel cache:',
    CHANNEL_CACHE
)


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def nested(
    page,
    names,
):

    roots = [
        SOURCE
        / 'whole_pages'
        / page,

        SOURCE
        / 'pages'
        / page,

        SOURCE
        / page,
    ]

    for root in roots:

        for name in names:

            path = (
                root
                / name
            )

            if path.exists():

                return path

    return None


def cached_channel(
    page,
    names,
):

    roots = [
        CHANNEL_CACHE
        / page,

        CHANNEL_CACHE,
    ]

    candidates = []

    for root in roots:

        for name in names:

            candidates.extend(
                [
                    root
                    / f'{name}.npy',

                    root
                    / f'{page}_{name}.npy',
                ]
            )

    for path in candidates:

        if path.exists():

            return path

    return None


# ------------------------------------------------------------
# Resolve all page image/array sources
# ------------------------------------------------------------

files = {}
missing = []


for page in sorted(
    full.page.unique()
):

    page = str(
        page
    )


    # --------------------------------------------------------
    # OBSERVED
    #
    # Prefer the exact Notebook 88 cached observed array.
    # Fall back to a rendered Notebook 88 image, then ROOT/aligned.
    # --------------------------------------------------------

    observed = cached_channel(
        page,
        [
            'observed',
            'obs',
        ],
    )

    if observed is None:

        observed = nested(
            page,
            [
                'observed.png',
                'observed_page.png',
            ],
        )

    if observed is None:

        observed = obsmap.get(
            page
        )


    # --------------------------------------------------------
    # CLEAN TRUTH
    #
    # This does not need to be in Notebook 88's output directory.
    # The development clean reference already lives in source/clean.
    # --------------------------------------------------------

    clean = nested(
        page,
        [
            'clean.png',
            'clean_truth.png',
        ],
    )

    if clean is None:

        clean = cleanmap.get(
            page
        )


    # --------------------------------------------------------
    # A1 FALLBACK
    #
    # Prefer the exact cached A1 source used by Notebook 88.
    #
    # Search several plausible durable names because this is
    # an audit notebook and should resolve the actual stored
    # source rather than silently manufacture a new A1.
    # --------------------------------------------------------

    a1 = cached_channel(
        page,
        [
            'a1',
            'a1_source',
            'a1_fallback',
            'baseline',
            'corrected',
        ],
    )

    if a1 is None:

        a1 = nested(
            page,
            [
                'a1.png',
                'a1_fallback.png',
                'baseline.png',
                'v7_8.png',
            ],
        )


    # --------------------------------------------------------
    # Record resolution
    # --------------------------------------------------------

    if (
        observed is None
        or clean is None
        or a1 is None
    ):

        missing.append(
            {
                'page':
                    page,

                'observed':
                    str(
                        observed
                    )
                    if observed
                    else None,

                'clean':
                    str(
                        clean
                    )
                    if clean
                    else None,

                'a1':
                    str(
                        a1
                    )
                    if a1
                    else None,
            }
        )

        continue


    files[
        page
    ] = {
        'observed':
            Path(
                observed
            ),

        'clean':
            Path(
                clean
            ),

        'a1':
            Path(
                a1
            ),
    }


# ------------------------------------------------------------
# Resolution audit
# ------------------------------------------------------------

missing_df = pd.DataFrame(
    missing
)


missing_df.to_csv(
    AUDIT
    / 'missing_images.csv',
    index=False,
)


resolution_rows = []


for page, resolved_files in files.items():

    resolution_rows.append(
        {
            'page':
                page,

            'observed':
                str(
                    resolved_files[
                        'observed'
                    ]
                ),

            'clean':
                str(
                    resolved_files[
                        'clean'
                    ]
                ),

            'a1':
                str(
                    resolved_files[
                        'a1'
                    ]
                ),
        }
    )


pd.DataFrame(
    resolution_rows
).to_csv(
    AUDIT
    / 'image_resolution.csv',
    index=False,
)


require(
    not missing,
    (
        f"{len(missing)} pages still lack one or more "
        "required observed / clean / A1 sources. "
        f"See {AUDIT / 'missing_images.csv'}."
    ),
)


print(
    'Image sets:',
    len(
        files
    )
)

print(
    'Observed:',
    len(
        files
    )
)

print(
    'Clean:',
    len(
        files
    )
)

print(
    'A1:',
    len(
        files
    )
)



RuntimeError: 56 pages lack observed, clean or A1 images. See /content/drive/MyDrive/MusicAnnotationProject/notebook_89_a5_visual_audit/20261123_095620_notebook89_a5_visual_audit/audit/missing_images.csv.

## 3. Recreate threshold decisions and score them


In [ ]:
def run_filter(frame,selected):
    final=np.asarray(selected,bool).copy()
    for page,g in frame.groupby('page',sort=False):
        idx=g.index.to_numpy(); im=load_gray(files[page]['observed']); mask=np.zeros(im.shape,np.uint8)
        take=idx[final[idx]]
        if not len(take): continue
        mask[frame.loc[take,'y'].to_numpy(),frame.loc[take,'x'].to_numpy()]=1
        _,lab,stats,_=cv2.connectedComponentsWithStats(mask,8)
        ids=lab[frame.loc[take,'y'].to_numpy(),frame.loc[take,'x'].to_numpy()]
        final[take[stats[ids,cv2.CC_STAT_AREA]<MIN_CLEAN_RUN]]=False
    return final
for t in THRESHOLDS: full[f'sel_{t:.3f}']=run_filter(full,full.prob.to_numpy()>=t)
full['first_threshold']=np.nan
for t in THRESHOLDS: full.loc[full[f'sel_{t:.3f}']&full.first_threshold.isna(),'first_threshold']=t

# Components are connected eligible regions, providing a stable crop identity across thresholds.
compmaps={}; statsmap={}
for page,g in full.groupby('page'):
    im=load_gray(files[page]['observed'])
    m=np.zeros(im.shape,np.uint8); 
    m[g.y,g.x]=1
    _,lab,stats,_=cv2.connectedComponentsWithStats(m,8); compmaps[page]=lab; statsmap[page]=stats

S=[]; P=[]; C=[]
for t in THRESHOLDS:
    col=f'sel_{t:.3f}'; s=full[col]
    S.append({'threshold':t,'selected':int(s.sum()),'annotation_cleaned':int((s&(full.truth==0)).sum()),'print_lost':int((s&(full.truth==1)).sum()),'truth_uncertain':int((s&(full.truth==2)).sum()),'coverage':float(s.mean())})
    for page,g in full.groupby('page'):
        q=g[g[col]].copy(); good=int((q.truth==0).sum()); bad=int((q.truth==1).sum()); unc=int((q.truth==2).sum())
        P.append({'threshold':t,'page':page,'source_group':g.source_group.iloc[0],'selected':len(q),'annotation_cleaned':good,'print_lost':bad,'truth_uncertain':unc,'utility':good-5*bad})
        if q.empty: continue
        q['component_id']=compmaps[page][q.y,q.x]
        for cid,z in q[q.component_id>0].groupby('component_id'):
            good=int((z.truth==0).sum()); bad=int((z.truth==1).sum()); unc=int((z.truth==2).sum()); cat=bad>=50 or (bad/max(good,1)>1 and bad>=20); st=statsmap[page][int(cid)]
            C.append({'threshold':t,'page':page,'source_group':z.source_group.iloc[0],'component_id':int(cid),'selected':len(z),'annotation_cleaned':good,'print_lost':bad,'truth_uncertain':unc,'catastrophic':cat,'utility':good-5*bad-(500 if cat else 0),'x':int(st[0]),'y':int(st[1]),'w':int(st[2]),'h':int(st[3])})
summary=pd.DataFrame(S); summary['print_loss_per_annotation_cleaned']=summary.print_lost/np.maximum(summary.annotation_cleaned,1)
pages=pd.DataFrame(P); components=pd.DataFrame(C)
summary.to_csv(TABLES/'threshold_summary.csv',index=False); pages.to_csv(TABLES/'results_by_page.csv',index=False); components.to_csv(TABLES/'results_by_component.csv',index=False)
full.to_parquet(TABLES/'threshold_decisions.parquet',index=False)
display(summary)


## 4. Generate high-resolution whole-page and component visualisations


In [ ]:
# ============================================================
# 4. High-resolution whole-page and component visual audit
#
# MEMORY-SAFE VERSION
#
# Truth semantics from Notebook 88:
#
#     0 = PRINT_KEEP
#     1 = ANNOTATION_CLEAN
#     2 = UNCERTAIN
#
# Colour semantics:
#
#     GREEN  = correct annotation cleanup
#     RED    = genuine print lost
#     PURPLE = truth uncertain
#
# A5 assembly:
#
#     start from exact A1
#     replace CLEAN-selected pixels with the frozen paper estimate
# ============================================================

import gc
import math

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# ============================================================
# Visual controls
# ============================================================

GREEN = (
    35,
    190,
    70,
)

RED = (
    20,
    20,
    240,
)

PURPLE = (
    210,
    40,
    210,
)


TC = {
    0.995:
        (
            220,
            70,
            20,
        ),

    0.990:
        (
            255,
            190,
            0,
        ),

    0.980:
        (
            0,
            215,
            255,
        ),

    0.950:
        (
            0,
            110,
            255,
        ),
}


# ============================================================
# Basic visual helpers
# ============================================================

def dil(
    mask,
    radius=REVIEW_RADIUS,
):

    mask = np.asarray(
        mask,
        dtype=bool,
    )

    if not radius:
        return mask.copy()

    kernel = np.ones(
        (
            2 * radius + 1,
            2 * radius + 1,
        ),
        dtype=np.uint8,
    )

    output = (
        cv2.dilate(
            mask.astype(
                np.uint8
            ),
            kernel,
        )
        > 0
    )

    del kernel

    return output


def overlay(
    gray,
    layers,
):

    base = cv2.cvtColor(
        gray,
        cv2.COLOR_GRAY2BGR,
    )

    colour = base.copy()

    active = np.zeros(
        gray.shape,
        dtype=bool,
    )

    for mask, bgr in layers:

        mask = np.asarray(
            mask,
            dtype=bool,
        )

        colour[
            mask
        ] = bgr

        active |= mask

    mixed = cv2.addWeighted(
        base,
        1.0
        - OVERLAY_ALPHA,
        colour,
        OVERLAY_ALPHA,
        0,
    )

    base[
        active
    ] = mixed[
        active
    ]

    del colour
    del mixed
    del active

    return base


def savegrid(
    images,
    titles,
    path,
    title,
    cols=4,
    dpi=170,
):

    require(
        len(images)
        == len(titles),
        (
            "Image/title count mismatch: "
            f"{len(images)} images versus "
            f"{len(titles)} titles"
        ),
    )

    rows = math.ceil(
        len(images)
        / cols
    )

    fig, axes = plt.subplots(
        rows,
        cols,
        figsize=(
            5 * cols,
            5 * rows,
        ),
        squeeze=False,
    )

    for axis in axes.flat:

        axis.axis(
            "off"
        )

    for axis, image, panel_title in zip(
        axes.flat,
        images,
        titles,
    ):

        display_image = image

        if (
            display_image.ndim
            == 3
        ):

            display_image = cv2.cvtColor(
                display_image,
                cv2.COLOR_BGR2RGB,
            )

            axis.imshow(
                display_image
            )

        else:

            axis.imshow(
                display_image,
                cmap="gray",
                vmin=0,
                vmax=255,
            )

        axis.set_title(
            panel_title,
            fontsize=9,
        )

        axis.axis(
            "off"
        )

        if (
            display_image
            is not image
        ):

            del display_image

    fig.suptitle(
        title,
        fontsize=12,
    )

    fig.tight_layout(
        rect=[
            0,
            0,
            1,
            0.98,
        ]
    )

    fig.savefig(
        path,
        dpi=dpi,
        bbox_inches="tight",
    )

    plt.close(
        fig
    )

    del axes
    del fig


# ============================================================
# Paper-estimate resolver
#
# Notebook 88 A5 CLEAN pixels are replaced with the frozen paper
# estimate, not with the observed annotated pixel.
# ============================================================

def resolve_paper_path(page):

    page = str(
        page
    )

    # Preferred: paper was explicitly added to the resolved file
    # record in the earlier image-resolution cell.
    paper = files[
        page
    ].get(
        "paper"
    )

    if (
        paper is not None
        and Path(
            paper
        ).exists()
    ):

        return Path(
            paper
        )

    # Common Notebook 88 arrangement:
    #
    # observable_channel_cache/<page>/a1.npy
    # observable_channel_cache/<page>/paper.npy
    a1_path = Path(
        files[
            page
        ][
            "a1"
        ]
    )

    sibling = (
        a1_path.parent
        / "paper.npy"
    )

    if sibling.exists():

        return sibling

    # Fall back to the CHANNEL_CACHE object resolved earlier.
    if (
        "CHANNEL_CACHE"
        in globals()
        and CHANNEL_CACHE is not None
    ):

        candidate = (
            Path(
                CHANNEL_CACHE
            )
            / page
            / "paper.npy"
        )

        if candidate.exists():

            return candidate

    raise RuntimeError(
        (
            f"{page}: could not resolve the frozen "
            "Notebook 88 paper estimate. "
            "Expected paper.npy beside the cached A1 array "
            "or under CHANNEL_CACHE/<page>/paper.npy."
        )
    )


# ============================================================
# Truth masks for one threshold
#
# The function no longer reloads the observed image.
# ============================================================

def masks(
    page,
    group,
    threshold,
    shape,
    region=None,
):

    selected = (
        group[
            f"sel_{threshold:.3f}"
        ].to_numpy(
            dtype=bool,
            copy=False,
        )
    )

    ys = group.y.to_numpy(
        dtype=np.int32,
        copy=False,
    )

    xs = group.x.to_numpy(
        dtype=np.int32,
        copy=False,
    )

    truth = group.truth.to_numpy(
        dtype=np.int8,
        copy=False,
    )

    good = np.zeros(
        shape,
        dtype=bool,
    )

    bad = np.zeros(
        shape,
        dtype=bool,
    )

    uncertain = np.zeros(
        shape,
        dtype=bool,
    )

    # Notebook 88:
    #
    # truth == 1 means ANNOTATION_CLEAN.
    good_hit = (
        selected
        & (
            truth
            == 1
        )
    )

    good[
        ys[
            good_hit
        ],
        xs[
            good_hit
        ],
    ] = True

    # Notebook 88:
    #
    # truth == 0 means PRINT_KEEP.
    bad_hit = (
        selected
        & (
            truth
            == 0
        )
    )

    bad[
        ys[
            bad_hit
        ],
        xs[
            bad_hit
        ],
    ] = True

    uncertain_hit = (
        selected
        & (
            truth
            == 2
        )
    )

    uncertain[
        ys[
            uncertain_hit
        ],
        xs[
            uncertain_hit
        ],
    ] = True

    if region is not None:

        require(
            region.shape
            == shape,
            (
                f"{page}: component-region shape "
                f"{region.shape} != page shape {shape}"
            ),
        )

        good &= region
        bad &= region
        uncertain &= region

    return (
        good,
        bad,
        uncertain,
    )


# ============================================================
# A5 candidate assembly
#
# A5 begins from exact A1 and replaces only CLEAN-selected pixels
# with the frozen paper estimate.
# ============================================================

def candidate(
    group,
    threshold,
    a1,
    paper,
):

    require(
        a1.shape
        == paper.shape,
        (
            "A1/paper shape mismatch: "
            f"{a1.shape} != {paper.shape}"
        ),
    )

    output = a1.copy()

    selected = (
        group[
            f"sel_{threshold:.3f}"
        ].to_numpy(
            dtype=bool,
            copy=False,
        )
    )

    ys = group.y.to_numpy(
        dtype=np.int32,
        copy=False,
    )

    xs = group.x.to_numpy(
        dtype=np.int32,
        copy=False,
    )

    output[
        ys[
            selected
        ],
        xs[
            selected
        ],
    ] = paper[
        ys[
            selected
        ],
        xs[
            selected
        ],
    ]

    return output


# ============================================================
# Crop geometry
# ============================================================

def cropbox(
    shape,
    x,
    y,
    width,
    height,
):

    image_height, image_width = shape

    side = min(
        MAX_CROP_SIDE,
        max(
            MIN_CROP_SIDE,
            width
            + 2
            * CROP_PADDING,
            height
            + 2
            * CROP_PADDING,
        ),
    )

    centre_x = (
        x
        + width
        // 2
    )

    centre_y = (
        y
        + height
        // 2
    )

    x0 = max(
        0,
        min(
            max(
                image_width
                - side,
                0,
            ),
            centre_x
            - side
            // 2,
        ),
    )

    y0 = max(
        0,
        min(
            max(
                image_height
                - side,
                0,
            ),
            centre_y
            - side
            // 2,
        ),
    )

    return (
        x0,
        y0,
        min(
            image_width,
            x0
            + side,
        ),
        min(
            image_height,
            y0
            + side,
        ),
    )


# ============================================================
# Whole-page comparison and

## 5. Build review queues, contact sheets and upload package


In [ ]:
queues={
 'catastrophic':crop_index[crop_index.ever_catastrophic].sort_values(['max_print_lost','max_annotation_cleaned'],ascending=[False,False]),
 'false_heavy':crop_index.sort_values(['max_print_lost','max_annotation_cleaned'],ascending=[False,False]),
 'beneficial':crop_index[crop_index.max_annotation_cleaned>0].sort_values(['max_print_lost','max_annotation_cleaned'],ascending=[True,False])}
# Threshold-boundary ranking: print loss newly entering at 0.950, plus useful gain by 0.980.
piv=components.pivot_table(index=['page','component_id'],columns='threshold',values=['annotation_cleaned','print_lost'],fill_value=0)
br=[]
for (page,cid),r in piv.iterrows():
    get=lambda metric,t:int(r[(metric,t)]) if (metric,t) in r.index else 0
    br.append({'page':page,'component_id':cid,'safe_gain_to_0980':get('annotation_cleaned',.980)-get('annotation_cleaned',.995),'new_loss_at_0950':get('print_lost',.950)-get('print_lost',.980)})
queues['threshold_boundary']=pd.DataFrame(br).merge(crop_index,on=['page','component_id']).sort_values(['new_loss_at_0950','safe_gain_to_0980'],ascending=[False,False])
for name,q in queues.items():
    q.head(MAX_QUEUE).to_csv(TABLES/f'{name}_review_queue.csv',index=False)
    ims=[]; titles=[]
    for _,r in q.head(24).iterrows():
        im=cv2.imread(str(r.crop_path));
        if im is not None: ims.append(im); titles.append(f'{r.page} C{int(r.component_id)} good={int(r.max_annotation_cleaned)} lost={int(r.max_print_lost)}')
    if ims: savegrid(ims,titles,VISUAL/f'{name}_contact_sheet.png',name.replace('_',' ').title(),4,120)

plt.figure(figsize=(9,6)); plt.plot(summary.print_loss_per_annotation_cleaned,summary.coverage,marker='o')
for _,r in summary.iterrows(): plt.annotate(f'{r.threshold:.3f}',(r.print_loss_per_annotation_cleaned,r.coverage))
plt.xlabel('Print lost per annotation pixel cleaned'); plt.ylabel('Eligible CLEAN coverage'); plt.title('A5 safety / coverage frontier'); plt.tight_layout(); plt.savefig(VISUAL/'safety_coverage_frontier.png',dpi=180); plt.close()

env={'python':sys.version,'opencv':cv2.__version__,'numpy':np.__version__,'pandas':pd.__version__,'platform':platform.platform()}; (OUT/'software_environment.json').write_text(json.dumps(env,indent=2))
identity={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'notebook89-a5-visual-audit','development_only':True,'promotion_authority':False,'retrained':False,'refitted':False,'recalibrated':False,'source_run':str(SOURCE),'source_predictions_sha256':sha256_file(PRED),'thresholds':THRESHOLDS,'review_radius':REVIEW_RADIUS,'pages':int(full.page.nunique()),'rows':int(len(full)),'component_crops':int(len(crop_index)),'software':env}; (OUT/'experiment_identity.json').write_text(json.dumps(identity,indent=2))
text='NOTEBOOK 89 A5 VISUAL AUDIT\n\nSource: '+str(SOURCE)+'Retrained: False\nRefitted: False\nRecalibrated: False\n\n'+summary.to_string(index=False)+'\n\nGreen=annotation cleaned; red=print lost; purple=truth uncertain.\n'
(OUT/'executive_summary.txt').write_text(text)

curated=[OUT/'executive_summary.txt',OUT/'experiment_identity.json',OUT/'software_environment.json',TABLES/'threshold_summary.csv',TABLES/'results_by_page.csv',TABLES/'results_by_component.csv',TABLES/'component_crop_index.csv',VISUAL/'safety_coverage_frontier.png']+[TABLES/f'{x}_review_queue.csv' for x in queues]+[VISUAL/f'{x}_contact_sheet.png' for x in queues]
for p in curated:
    if p.exists():
        dst=UPLOAD/p.relative_to(OUT); dst.parent.mkdir(parents=True,exist_ok=True); shutil.copy2(p,dst)
(UPLOAD/'README.txt').write_text('Notebook 89 review bundle. Start with executive_summary.txt and visual_audit contact sheets. Full-resolution crops remain in the run directory. Green=good cleanup; red=print loss; purple=uncertain.\n')
manifest=[]
for p in sorted(UPLOAD.rglob('*')):
    if p.is_file(): manifest.append({'relative_path':str(p.relative_to(UPLOAD)),'bytes':p.stat().st_size,'sha256':sha256_file(p)})
pd.DataFrame(manifest).to_csv(UPLOAD/'upload_manifest.csv',index=False)
zip_path=UPLOAD/'notebook89_upload_bundle.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(UPLOAD.rglob('*')):
        if p.is_file() and p!=zip_path: z.write(p,p.relative_to(UPLOAD))
print(text); print('Upload bundle:',zip_path)


## Inspect first

- `visual_audit/catastrophic_contact_sheet.png`
- `visual_audit/false_heavy_contact_sheet.png`
- `visual_audit/beneficial_contact_sheet.png`
- `visual_audit/threshold_boundary_contact_sheet.png`
- `tables/*_review_queue.csv`
- `crops/<page>/<page>_component_<id>.png`
- `whole_pages/<page>/whole_page_review.png`
- `upload/notebook89_upload_bundle.zip`
